In [2]:
from utils import readJson, writeJson


In [48]:
player_meta = readJson(f'Dataset/transfermarkt_fbref_dataset.json')
player_meta_manc = readJson(f'Dataset/transfermarkt_fbref_mancanti.json')
#merge giocatori mancanti con dataset originale
for i in range(len(player_meta_manc)):
    for j in range(len(player_meta)):
        if player_meta_manc[i]['stats'] != {} and player_meta_manc[i]['id'] == player_meta[j]['id']:
            player_meta[j] = player_meta_manc[i]

related_pos = readJson('Dataset/related_positions.json')
pos_translation = readJson('Dataset/tm_position_translation.json')
team_dict = {v: k for k, v in readJson('Dataset/merged_teams.json').items()}

In [22]:
kg_dir = 'Dataset/Knowledge_graph'

In [73]:
def get_player_entity(p, related, translation):
    pos = p['tm_role']
    pos_trans = translation[pos]
    pos_related = related['en'][pos_trans]
    d = {'Type': 'Player','id': p['id'], 'name': p['name']}
    d['preferred_position'] = pos_trans
    d['possible_position'] = pos_related
    d['market_value'] = p['tm_market_value']
    return d

def get_player_relationships(p):
    relationships = []
    stats_tables = p['stats']
    for stats_table in stats_tables.keys():
        stats = stats_tables[stats_table]
        if stats not in ['Standard Stats', 'Miscellaneous Stats']:
            for st, v in stats.items():
                if int(v['percentile']) <= 30:
                    d = {'from': p['id'], 'to': st, 'type': 'weak_in'}   
                elif int(v['percentile']) <= 49:
                    d = {'from': p['id'], 'to': st, 'type': 'average_in'}    
                elif int(v['percentile']) <= 74:
                    d = {'from': p['id'], 'to': st, 'type': 'good_in'}  
                elif int(v['percentile']) <= 89:
                    d = {'from': p['id'], 'to': st, 'type': 'very_good_in'}
                else:
                    d = {'from': p['id'], 'to': st, 'type': 'excellent_in'}
                
                relationships.append(d)

    d = {'from': p['id'], 'to': p['team'], 'type': 'play_for'}
    relationships.append(d)
    return relationships


# Costruzione entità skills

In [31]:
stat_nodes = {'nodes': []}
p = player_meta[0]
stats_k = []
for stat_table in p['stats'].keys():

    if stat_table not in ['Standard Stats', 'Miscellaneous Stats']:
        stats = p['stats'][stat_table]
        for st in stats.keys():
            if st not in stats_k:
                stats_k.append(st)
                d = {'type': 'skill', 'name': st, 'description': stats[st]['description']}
                stat_nodes['nodes'].append(d)
writeJson(stat_nodes, f'{kg_dir}/skill_entities.json')          

#  Costruzione entità squadre

In [37]:
teams = []
for p in player_meta:
    teams.append(p['team'])
teams = set(teams)
team_nodes = {'nodes': []}
for t in teams:
    d = {'type': 'Team', 'name': t, 'tm_name': team_dict[t]}
    team_nodes['nodes'].append(d)

writeJson(team_nodes, f'{kg_dir}/team_entities.json')

# Costruzione entità giocatori

In [78]:
player_entities = {'nodes': []}
for p in player_meta:
    if p['tm_role'] != 'Portiere' and p['stats'] != {}:
        p_ent = get_player_entity(p, related_pos, pos_translation)
        player_entities['nodes'].append(p_ent)

writeJson(player_entities, f'{kg_dir}/player_entities.json')
len(player_entities['nodes'])

1835

# Costruzione relazioni tra entità <br>
- Excellent (≥90th percentile) -> Major strength <br>
- Very Good (75-89th percentile) -> Significant asset <br>
- Good (50-74th percentile) -> Competent ability <br>
- Average (30-49th percentile) -> Room for improvement <br>
- Weak (<30th percentile) -> Notable weakness

In [97]:
rels = {'relationships': []}
for p in player_meta:
    if p['tm_role'] != 'Portiere' and p['stats'] != {}:
        r = get_player_relationships(p)
        if len(r) == 131:
            print(p['name'])
        rels['relationships'] = rels['relationships'] + r

writeJson(rels, f'{kg_dir}/relationships.json')
len(rels['relationships'])

Nicolas Seiwald
Santiago Bueno
Seamus Coleman
Luca Kilian
Dominique Heintz
Lisandro Martinez
Leonardo Bonucci
Thomas Isherwood
Connor Roberts
Jackson Tchatchoua
Triantafyllos Pasaridis
Stanley NSoki
Tim Drexler


249308

In [ ]:
r1 = []
r2 = []
for p in player_meta:
    if p['name'] == 'Jackson Tchatchoua':
        r1 = get_player_relationships(p)
        
    elif p['name'] == 'Virgil van Dijk':
        r2 = get_player_relationships(p)
       
to_r1 = [r['to'] for r in r1]
to_r2 = [r['to'] for r in r2]
for r in to_r2:
    if r not in to_r1:
        print(r)


{'from': 'aeca7743', 'to': 'Hellas Verona', 'type': 'play_for'}
Shots on Target %
Goals/Shot
Goals/Shot on Target
Average Shot Distance
npxG/Shot
Liverpool
